## 실습 5: AgentCore Evaluations - 고객 지원 Agent 온라인 평가

### 개요

이 실습에서는 AgentCore Evaluations를 사용하여 실습 4에서 배포한 프로덕션 고객 지원 Agent를 지속적으로 모니터링하는 방법을 살펴봅니다. 고객이 Agent와 상호 작용할 때 실시간으로 Agent 성능을 자동 평가하도록 온라인 평가를 구성합니다.

**워크숍 진행 과정:**

- **실습 1(완료):** Agent 프로토타입 만들기 - 작동하는 고객 지원 Agent 구축
- **실습 2(완료):** Memory로 기능 강화 - 대화 컨텍스트 및 개인화 추가
- **실습 3(완료):** Gateway 및 Identity로 확장 - Agent 간에 도구를 안전하게 공유
- **실습 4(완료):** 프로덕션에 배포 - AgentCore Runtime 및 Observability 사용
- **실습 5(현재):** Agent 성능 평가 - 온라인 평가를 통해 품질 모니터링
- **실습 6:** 사용자 인터페이스 구축 - 고객용 애플리케이션 만들기

### 학습 내용

내장 evaluator로 온라인 평가를 구성하고, 테스트 상호 작용을 생성하며, AgentCore Observability 대시보드에서 품질 지표를 분석하여 Agent 성능을 개선합니다.

### 온라인 평가 개요

특정 상호 작용을 선택하여 분석하는 온디맨드 평가와 달리 온라인 평가는 프로덕션에 배포된 Agent를 지속적으로 모니터링합니다. 구성 가능한 규칙을 사용하는 세션 샘플링, 다양한 평가 방식(내장 또는 사용자 지정 evaluator), 품질 추세와 낮은 점수의 세션을 조사할 수 있는 대시보드 모니터링의 세 가지 구성 요소로 이루어집니다.

Agent가 AgentCore Runtime에서 실행되므로 AgentCore Observability는 코드를 자동으로 계측하고 [OTEL](https://opentelemetry.io/) 계측을 사용하여 종합적인 로그와 추적을 제공합니다.

### 사전 요구 사항

실습 4를 완료하여 고객 지원 Agent를 배포해야 합니다. Evaluations 권한이 있는 AWS 계정으로 Amazon Bedrock AgentCore에 액세스할 수 있어야 합니다.

### 아키텍처
<div style="text-align:left">
    <img src="images/architecture_lab5_evaluation.png" width="75%"/>
</div>

*온라인 평가는 Agent 상호 작용을 자동으로 모니터링하고, 샘플링 규칙에 따라 evaluator를 적용하며, 분석할 수 있도록 결과를 CloudWatch에 출력합니다.*

### 단계 1: 필수 라이브러리 가져오기 및 클라이언트 초기화

In [ ]:
from bedrock_agentcore_starter_toolkit import Evaluation, Runtime
import json
import uuid
from pathlib import Path
from boto3.session import Session
from IPython.display import Markdown, display
from lab_helpers.utils import get_ssm_parameter, get_or_create_cognito_pool

In [ ]:
boto_session = Session()
region = boto_session.region_name
print(f"Region: {region}")

In [ ]:
eval_client = Evaluation(region=region)
runtime_client = Runtime()

### 단계 2: 실습 4의 Agent 정보 가져오기

실습 4 배포 중 SSM Parameter Store에 저장한 고객 지원 Agent ARN을 가져옵니다.

In [ ]:
try:
    # SSM Parameter Store에서 Agent ARN 가져오기(실습 4에서 저장)
    agent_arn = get_ssm_parameter("/app/customersupport/agentcore/runtime_arn")

    # ARN에서 Agent ID 추출
    agent_id = agent_arn.split(":")[-1].split("/")[-1]

    # Runtime 클라이언트 구성 경로 설정
    runtime_client._config_path = Path.cwd() / ".bedrock_agentcore.yaml"

    print("Agent ID:", agent_id)
    print("Agent ARN:", agent_arn)
except Exception as e:
    raise Exception(
        f"""Missing agent information from Lab 4. Please run lab-04-agentcore-runtime.ipynb first. Error: {str(e)}"""
    )

### 단계 3: 온라인 평가 구성 생성

이제 고객 지원 Agent의 온라인 평가 구성을 생성합니다. 내장 evaluator를 사용하여 Agent 성능의 여러 측면을 평가합니다.

- **Builtin.GoalSuccessRate** - Agent가 사용자 목표를 얼마나 잘 달성하는지 측정
- **Builtin.Correctness** - 응답의 사실 정확성 평가
- **Builtin.ToolSelectionAccuracy** - 적절한 도구 선택 여부 평가

시연을 위해 샘플링 비율을 100%로 설정하지만, 프로덕션에서는 트래픽 양에 따라 더 낮은 비율(예: 10~20%)을 사용할 수 있습니다.

In [ ]:
response = eval_client.create_online_config(
    agent_id=agent_id,
    config_name="customer_support_agent_eval",
    sampling_rate=100,  # 데모에서는 session 100% 평가
    evaluator_list=[
        "Builtin.GoalSuccessRate",
        "Builtin.Correctness",
        "Builtin.ToolSelectionAccuracy",
    ],
    config_description="Customer support agent online evaluation",
    auto_create_execution_role=True,
)

print("Online evaluation configuration created successfully!")
print(f"Configuration ID: {response['onlineEvaluationConfigId']}")

### 단계 4: 구성 상태 확인

평가 구성의 세부 정보를 가져와 올바르게 생성되고 활성화되었는지 확인합니다.

In [ ]:
config_details = eval_client.get_online_config(config_id=response["onlineEvaluationConfigId"])
print("Configuration Details:")
print(json.dumps(config_details, indent=2, default=str))

### 단계 5: 테스트 상호 작용 생성

다양한 질의로 고객 지원 Agent를 호출하여 평가용 추적을 생성합니다. 여러 테스트 시나리오를 통해 evaluator가 Agent 성능을 평가하는 방식을 확인할 수 있습니다.

In [ ]:
# 인증 토큰 가져오기
access_token = get_or_create_cognito_pool(refresh_token=True)
print(f"Access token obtained: {access_token['bearer_token'][:20]}...")


def invoke_agent_runtime(prompt, session_id=None):
    """Starter Toolkit을 사용하여 에이전트 Runtime을 호출합니다."""
    if not session_id:
        session_id = str(uuid.uuid4())

    response = runtime_client.invoke(
        payload={"prompt": prompt},
        session_id=session_id,
        bearer_token=access_token["bearer_token"],
    )

    return response, session_id

#### 테스트 시나리오 1: 제품 정보 질의

In [ ]:
session1 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need information about the Gaming Console Pro. What are its specifications and price?",
    session1,
)
print("Customer Query: Product information request")
display(Markdown(response["response"].replace("\\n", "\n")))

#### 테스트 시나리오 2: 기술 지원 요청

In [ ]:
session2 = str(uuid.uuid4())
response, _ = invoke_agent_runtime("My laptop won't start up. Can you help me troubleshoot this issue?", session2)
print("Customer Query: Technical support request")
display(Markdown(response["response"].replace("\\n", "\n")))

#### 테스트 시나리오 3: 반품 정책 문의

In [ ]:
session3 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I bought a smartphone last week but it's not working properly. What's your return policy?",
    session3,
)
print("Customer Query: Return policy inquiry")
display(Markdown(response["response"].replace("\\n", "\n")))

#### 테스트 시나리오 4: 복잡한 다중 도구 질의

In [ ]:
session4 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "I need help with my Gaming Console Pro. First, can you tell me about its warranty? Then I need technical support for connection issues.",
    session4,
)
print("Customer Query: Complex multi-tool request")
display(Markdown(response["response"].replace("\\n", "\n")))

#### 테스트 시나리오 5: 일반 기능 질의

In [ ]:
session5 = str(uuid.uuid4())
response, _ = invoke_agent_runtime(
    "What kind of support can you provide? List all your available tools and capabilities.",
    session5,
)
print("Customer Query: Capability inquiry")
display(Markdown(response["response"].replace("\\n", "\n")))

### 단계 6: 평가 결과 모니터링

AgentCore Observability 콘솔에서 평가 결과를 모니터링합니다. 시스템에서 추적을 처리하고 evaluator를 적용하므로 결과가 표시되기까지 몇 분 정도 걸릴 수 있습니다.

#### 대시보드 액세스

1. [AgentCore Observability 콘솔](https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability/agent-core/agents)로 이동합니다.
2. Agent 목록에서 고객 지원 Agent를 찾습니다.
3. `DEFAULT` 엔드포인트를 클릭하여 평가 지표를 확인합니다.
4. 추적 및 세션 보기에서 평가 점수를 확인합니다.

#### 확인할 수 있는 항목

대시보드에는 다음 항목이 표시됩니다.
- **Goal Success Rate**: Agent가 고객 목표를 얼마나 잘 달성하는지 측정
- **Correctness**: 제공된 정보의 정확성
- **Tool Selection Accuracy**: 질의에 적합한 도구 선택 여부

![온라인 평가 대시보드](images/online_evaluations_dashboard.png)

*AgentCore Observability 대시보드에 표시된 평가 지표*

### 단계 7: 평가 지표 이해

**Goal Success Rate**는 Agent가 고객의 주요 의도를 성공적으로 해결하는지 측정합니다. 점수가 높으면 문제를 효과적으로 해결했다는 뜻이며, 점수가 낮으면 요구 사항을 충족하지 못했거나 응답이 불완전하거나 요청을 잘못 이해했을 수 있습니다.

**Correctness**는 응답의 사실 정확성을 평가합니다. 점수가 높으면 정보가 정확하고 신뢰할 수 있다는 뜻이며, 점수가 낮으면 잘못된 사실, 오래된 정보, 오해를 유발하는 안내가 포함되었을 수 있습니다.

**Tool Selection Accuracy**는 Agent가 각 작업에 적합한 도구를 선택하는지 평가합니다. 점수가 높으면 도구를 올바르게 선택했다는 뜻이며, 점수가 낮으면 잘못된 도구 선택, 불필요한 호출, 필요한 도구의 미사용이 발생했을 수 있습니다.

### 단계 8: 결과 분석 및 다음 단계

**Goal Success Rate가 낮은 경우:** Agent의 system prompt를 개선하고, 도구 설명과 파라미터를 보완하며, 구체적인 학습 예제를 추가합니다.

**Correctness 점수가 낮은 경우:** Knowledge Base를 최신 정보로 업데이트하고, 사실 확인 메커니즘을 개선하며, 도구 응답을 검토합니다.

**도구 관련 문제가 있는 경우:** 도구 파라미터 스키마를 개선하고, 도구 선택 로직을 보완하며, 도구 문서를 강화합니다.

**지속적인 모니터링:** 평가 지표에 대한 CloudWatch 경보를 설정하고, 추세 분석용 대시보드를 생성하며, 품질 저하에 대한 자동 알림을 구현합니다.

### 단계 9: 정리(선택 사항)

필요한 경우 아래 코드의 주석을 해제하여 온라인 평가 구성을 비활성화합니다.

In [ ]:
# 평가 구성을 비활성화하려면 다음 줄의 주석을 해제하세요.
# eval_client.delete_online_config(config_id=response['onlineEvaluationConfigId'])
# print("Online evaluation configuration disabled")

### 축하합니다! 🎉

**실습 5: AgentCore Evaluations - 온라인 평가**를 성공적으로 완료했습니다!

### 완료한 작업

Goal Success Rate(고객 만족도 및 문제 해결), Correctness(사실 정확성), Tool Selection Accuracy(올바른 도구 사용)를 평가하는 내장 evaluator를 사용하여 고객 지원 Agent의 자동 연속 온라인 평가를 구성했습니다. 평가 결과는 AgentCore Observability 대시보드와 통합되어 실시간 인사이트를 제공합니다.

**주요 이점:** 선제적 품질 보증을 통해 고객에게 영향을 주기 전에 문제를 발견하고, 데이터 기반 최적화로 개선 방향을 제시하며, 대규모 성능 모니터링으로 프로덕션 신뢰도를 높이고, 지속적인 학습을 통해 패턴과 기회를 식별할 수 있습니다.

**다음 단계:** 평가 대시보드를 정기적으로 모니터링하고, 품질 임계값에 대한 CloudWatch 경보를 설정하며, 인사이트를 활용해 Agent를 반복적으로 개선하고, 도메인별 지표를 위한 사용자 지정 evaluator 추가를 고려하세요.

### 다음 실습: [실습 6: 사용자 인터페이스 구축 →](lab-06-frontend.ipynb)

고객이 품질 모니터링이 적용된 Agent와 상호 작용할 수 있는 사용자 친화적인 웹 인터페이스를 구축하여 고객 경험을 완성합니다.

이제 고객 지원 Agent가 종합적인 품질 모니터링을 갖추고 프로덕션 환경에서 실행할 준비를 마쳤습니다! 🚀